# Практическая работа № 7. Случайный лес
**Уровень B.** Предсказываем размер чаевых (`tip`) по остальным доступным признакам. Сравниваем дерево и случайный лес на одинаковом разбиении, подбираем гиперпараметры кросс-валидацией и проверяем качество после удаления наименее важных признаков.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, precision_recall_curve, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, precision_score, recall_score, f1_score)
RANDOM_STATE = 42
DATA_PATH = Path("../data/tips.csv") if Path("../data/tips.csv").exists() else Path("data/tips.csv")
df = pd.read_csv(DATA_PATH)
print("Размер:", df.shape)
df.head()

In [ ]:
X = df.drop(columns=["tip"])
y = df["tip"]
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
tree = Pipeline([("preprocess", pre), ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
forest_default = Pipeline([("preprocess", pre), ("model", RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1))])
tree.fit(X_train, y_train); forest_default.fit(X_train, y_train)
pred_tree = tree.predict(X_test); pred_forest = forest_default.predict(X_test)
def metrics(y_true,pred):
    return {"MAE":mean_absolute_error(y_true,pred),"RMSE":mean_squared_error(y_true,pred)**0.5,"R2":r2_score(y_true,pred)}
display(pd.DataFrame({"Decision Tree":metrics(y_test,pred_tree),"Random Forest (default)":metrics(y_test,pred_forest)}).T)

In [ ]:
param_grid = {"model__n_estimators":[100,200], "model__max_depth":[None,5,10], "model__min_samples_leaf":[1,2,4]}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(Pipeline([("preprocess", pre), ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))]),
                      param_grid=param_grid, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1, refit=True)
search.fit(X_train, y_train)
print("Лучшие параметры:", search.best_params_)
print("Лучший CV MAE:", -search.best_score_)
pred_tuned = search.predict(X_test)
display(pd.DataFrame({"Tree":metrics(y_test,pred_tree),"Forest default":metrics(y_test,pred_forest),"Forest tuned":metrics(y_test,pred_tuned)}).T)

In [ ]:
best_pipe = search.best_estimator_
feature_names = best_pipe.named_steps["preprocess"].get_feature_names_out()
importances = best_pipe.named_steps["model"].feature_importances_
importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
display(importance.head(15).to_frame("importance"))
importance.head(15).sort_values().plot(kind="barh", figsize=(8,5), title="Топ-15 важностей признаков")
plt.xlabel("Важность"); plt.tight_layout(); plt.show()
print("Топ-2 признака:", importance.head(2).index.tolist())

In [ ]:
# Эксперимент: исключаем нижние 20% признаков по встроенной важности.
keep_count = max(1, int(np.ceil(len(importance)*0.8)))
keep_names = set(importance.sort_values(ascending=False).head(keep_count).index)
# В исходном X удаляем признаки, чьи закодированные варианты все попали в нижнюю часть важности.
keep_original = []
for col in X.columns:
    matches = [name for name in importance.index if name.split("__",1)[-1] == col or name.split("__",1)[-1].startswith(col+"_")]
    if not matches or any(name in keep_names for name in matches):
        keep_original.append(col)
if not keep_original:
    keep_original = X.columns.tolist()
Xr_train, Xr_test = X_train[keep_original], X_test[keep_original]
num_r = [c for c in num_cols if c in keep_original]; cat_r = [c for c in cat_cols if c in keep_original]
pre_r = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_r),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_r)
])
reduced_model = Pipeline([("preprocess", pre_r), ("model", RandomForestRegressor(**{k.replace("model__",""):v for k,v in search.best_params_.items()}, random_state=RANDOM_STATE, n_jobs=-1))])
reduced_model.fit(Xr_train,y_train); pred_r=reduced_model.predict(Xr_test)
print("Оставлены исходные признаки:", keep_original)
display(pd.DataFrame({"Настроенный лес":metrics(y_test,pred_tuned),"Лес после сокращения":metrics(y_test,pred_r)}).T)

## Вывод
Число деревьев выбрано через CV среди 100 и 200: увеличение ансамбля обычно стабилизирует прогноз, но требует больше времени; лучший вариант выбирается по MAE на кросс-валидации. Сравнение дерева, леса по умолчанию и настроенного леса выполнено на одинаковом test. Встроенная важность описывает вклад в снижение impurity, а не причинность; удалять признаки стоит только если качество на тесте практически не ухудшается.